<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11-vision-navigation/Ex11_vision_navigation.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11 — Vision-Based Navigation, from A to B

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L11.1 · Vision-Based Navigation and L11.2 · Obstacle Avoidance**

A small car has to drive from A to B along a taped line, past obstacles
standing on it, as fast as it safely can, deciding from its camera alone. The
two lectures ask four questions on the way, and this notebook answers each one
with a measurement:

| | the question | lecture |
|---|---|---|
| **where is the path?** | the position of the line a little way ahead, from one frame | L11.1 |
| **how do I steer onto it?** | the steering that follows, and how often it must be decided | L11.1 |
| **is the path blocked?** | free or blocked, from one frame | L11.2 |
| **what do I do about it?** | stop, steer around, or follow a potential field | L11.2 |

Everything runs here, on Colab or a laptop, **with no car**: the frames are
drawn by `problem.py`, so the true answer to every question is known, and the
car is a simulated one. The same task on the real JetRacer is a demonstration
in the lab; the car and its extra sensors are for the mini projects.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the task, the drawn frames, the classical line detector | — |
| **4** | a network that finds the line, trained on two collections of frames | TODO 1 — the network |
| **5** | a network that says free or blocked, and its threshold | TODO 2 — the cross-entropy |
| **6** | three avoidance strategies on the course from A to B | TODO 3 — the potential field |
| **7** | the loop rate, the stopping distance, the GPU and fewer bits | — |
| **8** | what the notebook says | — |
| **9 – 10** | your report, and the mini project proposals | your answers |

About a minute on a CPU. **Use a GPU runtime on Colab** (Runtime → Change
runtime type → T4 GPU): section 7 compares the two. Numbers are printed with at
most two decimals, except errors smaller than that.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11-vision-navigation/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import copy
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex11_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The task from A to B

The course is 6 m long, along a taped line from A to B. Three obstacles of
radius 8 cm stand on or beside the line, 1.5 m apart; the shaded ring round
each is the car's half-width, 10 cm, so the car's centre must stay outside it.
**The course, the car's size and how far its camera judges free or blocked
(0.7 m) are assumed**, chosen to be about the size of the lab's floor and the
JetRacer.

**The car** is the kinematic bicycle model: wheelbase $L$ = 0.20 m, steering
limit 30°, both assumed. Every decision, it aims at a point a distance
$\ell$ = 0.40 m ahead, at an angle $\alpha$ to its heading, and steers by
pure pursuit (L11.1, *Classical Pipeline*):

$$\delta = \arctan\!\Big(\frac{2L\sin\alpha}{\ell}\Big)$$

**Between two decisions the car is blind.** At speed $v$ and $f$ decisions a
second it drives

$$d = \frac{v}{f}$$

on its last command, with no new information (L11.1, *Why a GPU*).

The cell below prints the problem and draws the course.

In [ ]:
pb.describe_problem()
ax = pb.draw_course()
ax.set_title("the course from A to B: the taped line, three obstacles, and the car's half-width round each", fontsize=9)
plt.show()

**What you should see.** The numbers of the problem, and the course: the grey
line from A to B, three red obstacles, the first on the line, the other two a
few centimetres to either side of it.

---

## 2 · The drawn frames

The camera's frame is 48 × 64 grey pixels: the floor near the car at the
bottom, the floor further away at the top. **The frames are drawn by
`problem.py`, not taken by a camera**, which is what makes the true answer
known.

| | |
|---|---|
| frame | 48 × 64 pixels, grey, values 0 to 1 |
| floor | brightness 0.35, with a slope across the frame of up to ±0.15: light from one side |
| tape | 0.55 brighter than the floor, about 2 pixels wide where the line is read |
| noise | 0.04 on every pixel |
| glare | on some frames, a bright patch beside the line: a window reflected in the floor |
| obstacle | a dark box with a lighter front face and sharp edges, smaller the further away it stands |
| shadow | a soft dark patch with no edges: **not** an obstacle |

**Two questions are asked of a frame.**

* **Where is the path?** The position of the line in the row 0.6 of the way up
  the frame, as a number from −1 (the left edge) to +1 (the right edge): the
  **target**. It is a regression (L11.1, *Steering From a Frame*). Three ways
  of collecting frames are used below: along the line only (`line`); from
  beside it and turned away (`off-line`); and as off-line with glare on every
  frame (`glare`).
* **Is the path blocked?** 1 when an obstacle stands in the car's lane in the
  near part of the frame, 0 otherwise. It is a classification (L11.2, *Free or
  Blocked*).

The cell draws six frames with their target, and three with their label.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(13, 4.4))
for i, kind in enumerate(("line", "line", "off-line", "off-line", "glare", "glare")):
    img, tgt = pb.frames(1, kind, seed=20 + i)                # one frame, and its true target
    axes[0, i].imshow(img[0, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].plot((pb.W - 1) / 2 * (1 + tgt[0]), pb.ROW, "r+", ms=12, mew=2)   # the target, in pixels
    axes[0, i].set_title(f"{kind}: target {tgt[0]:+.2f}", fontsize=9)
xs, bs, ws = pb.obstacle_frames(400, seed=11)                 # frames with obstacles and shadows
show = [int(np.where((bs == 0) & np.isnan(ws[:, 0]))[0][0]),  # free: no obstacle at all
        int(np.where((bs == 1) & (ws[:, 0] < 0.3))[0][0]),    # blocked: a box near, in the lane
        int(np.where((bs == 0) & ~np.isnan(ws[:, 0]))[0][0])] # free: a box, but far or beside the lane
for j, k in enumerate(show):
    axes[1, j].imshow(xs[k, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title("blocked" if bs[k] else "free", fontsize=9)
for a in axes.flat:
    a.axis("off")
plt.tight_layout(); plt.show()

**What you should see.** In the top row, a red cross on the tape in row 19;
one pixel is 0.032 of the target's range. In the last two frames there is a
second bright thing, the glare, which is not the line. In the bottom row, a
free frame, a blocked one with a box close in front, and a free one whose box
stands too far away or beside the lane to matter yet. Some frames carry a soft
shadow; it is not an obstacle.

---

## 3 · The classical detector

**The classical way to find the line** needs no training (L11.1, *Classical
Pipeline*). Take a few rows round the look-ahead row, estimate the floor's
brightness by the median, keep what is brighter than the floor by 0.15, and
return the brightness-weighted mean column:

$$\hat{x} = \frac{\sum_j w_j\,x_j}{\sum_j w_j}, \qquad w_j = \max(I_j - I_{\mathrm{floor}} - 0.15,\ 0)$$

`pb.detect(images)` does that.

**Its cost is set by the resolution.** `pb.shrink(images, k)` replaces every
$k \times k$ block of pixels by its mean. A smaller frame is cheaper to read
out and to process, and blurs the tape.

**The agreed accuracy is 0.03**: a mean error of one pixel of the full frame.
The cell tests the detector at four resolutions on 1000 frames of each
collection.

In [ ]:
TEST = {kind: pb.frames(1000, kind, seed=100 + i) for i, kind in enumerate(("line", "off-line", "glare"))}   # held out: never trained on
AGREED = 0.03

def mean_error(predict, kind):
    x, y = TEST[kind]
    return float(np.mean(np.abs(predict(x) - y)))             # the mean distance from the true target

classic = {}
for k in (1, 2, 4, 8):
    classic[k] = [mean_error(lambda x: pb.detect(pb.shrink(x, k)), kind) for kind in TEST]
    ok = max(classic[k][:2]) <= AGREED
    print(f"  {pb.H // k:2d} x {pb.W // k:2d} pixels: line {classic[k][0]:.3f}, off-line {classic[k][1]:.3f}, glare {classic[k][2]:.3f}"
          + ("   <- meets 0.03 without glare" if ok else ""))
K = 4                                                         # the coarsest frame that meets it
frame1 = TEST["line"][0][:1]                                  # one frame, for the timing
t_classic = middle(lambda: pb.detect(pb.shrink(frame1, K)))
print(f"  the detector on one 12 x 16 frame: {1000 * t_classic:.2f} ms")

**What you should see.** Without glare the detector is within 0.002 at full
resolution and within 0.03 down to **12 × 16 pixels**; at 6 × 8 the tape is
blurred into the floor and it fails. It does not care where the car is: line
and off-line are the same to it. **With glare it is wrong by 0.25 to 0.46 at
every resolution**: the glare is bright, and bright is all it looks for
(L11.1, *Where Conventional Vision Fails*).

---

## 4 · The steering network and deliberate data

**The network** takes the frame and returns the target. Three convolutional
layers, each halving the frame and doubling the channels, then two dense
layers:

| layer | output |
|---|---|
| convolution 5 × 5, stride 2, 8 channels, ReLU | 8 × 24 × 32 |
| convolution 3 × 3, stride 2, 16 channels, ReLU | 16 × 12 × 16 |
| convolution 3 × 3, stride 2, 32 channels, ReLU | 32 × 6 × 8 |
| flatten, dense 32, ReLU | 32 |
| dense, `outputs` numbers | the answer |

**The same network answers both questions of the block**: with one output it
finds the line (here), with two it says free or blocked (section 5). A
convolution is the same small sum of products over every window of the frame
(L5.1), which is why it finds the tape wherever it lies.

**The loss** is the mean squared distance between the network's target and the
true one, averaged over the training frames (L11.1, *Steering From a Frame*).

**Two datasets, 3000 frames each.** One collected along the line only. One
collected **deliberately**: from beside the line and turned away from it, and
with glare on a third of the frames (L11.1, *Collect the Data Deliberately*).

**TODO 1** is the network.

In [ ]:
class Net(torch.nn.Module):                                   # a network: layers in __init__, the computation in forward
    def __init__(self, outputs):
        super().__init__()
        nn = torch.nn
        # TODO 1: the layers of the table above, in an nn.Sequential called self.body.
        #
        #   self.body = nn.Sequential(                            # the layers, applied one after the other
        #       nn.Conv2d(1, 8, 5, stride=2, padding=2), nn.ReLU(),     # convolution: 1 channel in, 8 out, the frame halved
        #       nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),    # convolution: 16 channels, halved again
        #       nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # convolution: 32 channels, 6 x 8 left
        #       nn.Flatten(),                                     # the 32 x 6 x 8 block as one row of 1536 numbers
        #       nn.Linear(32 * 6 * 8, 32), nn.ReLU(),             # dense layer: 1536 inputs -> 32 outputs
        #       nn.Linear(32, outputs))                           # dense layer: 32 inputs -> the answer
        raise NotImplementedError("Write the network's layers")
    def forward(self, x):
        return self.body(x)                                   # outputs numbers per frame

In [ ]:
def train(outputs, x, y, loss_fn, epochs, batch=64):
    # a network with `outputs` outputs, trained on the frames x and answers y
    set_seed(88)                                              # fix the random start so a run repeats
    net = Net(outputs).to(DEVICE)                             # the network, on the device
    X, Y = to_tensor(x).to(DEVICE), torch.as_tensor(y).to(DEVICE)   # NumPy arrays -> tensors on the device
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)         # Adam: adaptive steps for every weight
    t0 = time.perf_counter()
    for _ in range(epochs):                                   # one epoch: every frame seen once
        order = torch.randperm(len(X), device=DEVICE)         # the frames in a new random order
        for i in range(0, len(X), batch):
            b = order[i:i + batch]                            # one mini-batch of 64 frames
            opt.zero_grad()                                   # clear the gradients of the step before
            loss = loss_fn(net(X[b]), Y[b])                   # how far the answers are from the truth
            loss.backward()                                   # backpropagation: the gradient of the loss
            opt.step()                                        # one Adam step on the weights
    return net, time.perf_counter() - t0

def squared(out, y):
    return (out[:, 0] - y).pow(2).mean()                      # mean squared distance from the true target

def steer_net(kind):
    x, y = pb.frames(3000, kind, seed=1)                      # 3000 frames collected the way `kind` says
    return train(1, x, y, squared, epochs=12)

net_line, sec_line = steer_net("line")                        # collected along the line only
net, sec = steer_net("deliberate")                            # collected deliberately
print(f"{parameter_count(net)} weights; trained in {sec_line:.0f} s and {sec:.0f} s")

def run(model):
    # the steering network as a function of NumPy frames, for mean_error
    def predict(x):
        with torch.no_grad():  # no gradient tracking: evaluation only
            return to_numpy(model(to_tensor(x).to(DEVICE)))[:, 0]   # frames -> the network -> NumPy targets
    return predict

In [ ]:
err_line = [mean_error(run(net_line), kind) for kind in TEST]
err_net = [mean_error(run(net), kind) for kind in TEST]
x1 = to_tensor(frame1).to(DEVICE)                             # one frame, for the timing
with torch.no_grad():  # no gradient tracking: evaluation only
    t_net = middle(lambda: net(x1))

print(f"{'mean error on 1000 held-out frames':40s}{'line':>8s}{'off-line':>10s}{'glare':>8s}{'one frame':>12s}{'training':>10s}")
print(f"  {'classical detector, 12 x 16':38s}{classic[K][0]:8.3f}{classic[K][1]:10.3f}{classic[K][2]:8.3f}{1000 * t_classic:9.2f} ms{'-':>10s}")
print(f"  {'network, trained along the line':38s}{err_line[0]:8.3f}{err_line[1]:10.3f}{err_line[2]:8.3f}{1000 * t_net:9.2f} ms{sec_line:8.0f} s")
print(f"  {'network, trained deliberately':38s}{err_net[0]:8.3f}{err_net[1]:10.3f}{err_net[2]:8.3f}{1000 * t_net:9.2f} ms{sec:8.0f} s")

x, y = TEST["glare"]
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for a, (name, pred) in zip(ax, (("classical detector", pb.detect(pb.shrink(x, K))),
                                ("network, along the line", run(net_line)(x)), ("network, deliberately", run(net)(x)))):
    a.plot([-1, 1], [-1, 1], "k--", lw=1); a.plot(y, pred, ".", ms=2)
    a.set_title(name + ", frames with glare", fontsize=9); a.set_xlabel("true target")
ax[0].set_ylabel("predicted target"); plt.tight_layout()
plt.savefig(cc.output_path("Ex11_report_1.png"), dpi=150); plt.show()

**What you should see.** Three rows that fail in three different places.

* **The classical detector** is within 0.03 without glare and wrong by about
  0.37 with it.
* **The network trained along the line** is within 0.002 on frames like the
  ones it saw, and wrong by about 0.14 the moment the car is beside the line -
  which is exactly when the answer matters. It has never seen the line
  anywhere but in the middle.
* **The network trained deliberately** is within 0.01 on all three. Same
  network, same number of frames, same training time. Only the collection
  differs.

In the plots, the detector's points scatter far from the diagonal wherever the
glare outweighed the tape; the first network's flatten towards the middle; the
second's lie on it. That answers *where is the path*.

---

## 5 · The obstacle network

**Is the path blocked?** The classical answer is a rule: in the lower half of
the frame, in the lane, is anything darker than the floor by 0.12?
`pb.dark_rule(images)` does that. A shadow is dark too, which is where it
fails (L11.2, *Identifying an Obstacle*).

**The network** is section 4's with two outputs, $z_{\mathrm{f}}$ for free and
$z_{\mathrm{b}}$ for blocked. The softmax turns them into the probability that
the path is blocked (L11.2, *Free or Blocked*):

$$p_{\mathrm{b}} = \frac{e^{z_{\mathrm{b}}}}{e^{z_{\mathrm{f}}} + e^{z_{\mathrm{b}}}}$$

and the loss is the **cross-entropy**, the mean over a batch of $N$ frames of
minus the log-probability the network gives the true label $y_k$:

$$\mathcal{L} = -\frac{1}{N}\sum_{k=1}^{N}\left[y_k \ln p_{\mathrm{b},k} + (1 - y_k)\ln(1 - p_{\mathrm{b},k})\right]$$

It is trained on 4000 frames from `pb.obstacle_frames`, four in five with a
box somewhere and two in five with a shadow, and tested on 2000 others.

**TODO 2** is the cross-entropy. Compute it from the log of the softmax,
`torch.log_softmax`, which is exact where the softmax itself would round a
tiny probability to zero.

In [ ]:
def cross_entropy(out, y):
    # out: the two outputs per frame (N, 2); y: the true label, 0 free or 1 blocked (N,)
    # TODO 2: the cross-entropy of the equation above.
    #
    #   logp = torch.log_softmax(out, dim=1)                      # ln p_f and ln p_b of every frame, by the softmax
    #   return -logp[torch.arange(len(y)), y].mean()              # minus the mean log-probability of the true label
    raise NotImplementedError("Write the cross-entropy")

In [ ]:
xo, bo, _ = pb.obstacle_frames(4000, seed=1)                  # training frames: obstacles, shadows, and their labels
obs_net, sec_obs = train(2, xo, bo, cross_entropy, epochs=15)
xt, bt, wt = pb.obstacle_frames(2000, seed=2)                 # held out: never trained on
with torch.no_grad():  # no gradient tracking: evaluation only
    p_blocked = to_numpy(torch.softmax(obs_net(to_tensor(xt).to(DEVICE)), 1))[:, 1]   # the softmax's probability of blocked
shadow_only = (bt == 0) & np.isnan(wt[:, 0])                  # frames with no box at all; some carry a shadow

def rates(pred):
    # right, missed obstacles (blocked called free), false stops (free called blocked)
    return (float(np.mean(pred == bt)), float(np.mean(pred[bt == 1] == 0)), float(np.mean(pred[bt == 0] == 1)))

rule = pb.dark_rule(xt)
cnn = (p_blocked > 0.5).astype(int)
print(f"{'2000 held-out frames':30s}{'right':>8s}{'missed':>9s}{'false stops':>13s}{'frames with no box called blocked':>36s}")
for name, pred in (("dark-pixel rule", rule), ("network, threshold 0.5", cnn)):
    r = rates(pred)
    print(f"  {name:28s}{100 * r[0]:7.1f}%{100 * r[1]:8.1f}%{100 * r[2]:12.1f}%{100 * pred[shadow_only].mean():35.1f}%")
print(f"network trained in {sec_obs:.0f} s")

**What you should see.** The rule is right on about 84 % of the frames: it
misses about a fifth of the obstacles, because a box's lighter front face is
not dark enough, and it calls about one frame in seven with no box blocked,
because of the shadows. The network is right on about 97.5 % and misses none.

**The threshold.** The car treats the path as blocked when $p_{\mathrm{b}}$ is
above a threshold $\tau$. One half is not the right value, because the two
errors do not cost the same: a missed obstacle is a collision, a false stop a
second lost. With $c_{\mathrm{miss}}$ and $c_{\mathrm{stop}}$ the costs, the
expected cost of a threshold is (L11.2, *The Cost of an Error*)

$$C(\tau) = c_{\mathrm{miss}}\,P_{\mathrm{miss}}(\tau) + c_{\mathrm{stop}}\,P_{\mathrm{stop}}(\tau)$$

The cell sweeps $\tau$ and takes the cheapest, with a collision costed **100
times** a false stop - an assumption, to be argued in your report.

In [ ]:
C_MISS, C_STOP = 100.0, 1.0                                   # ASSUMED: a collision costs 100 false stops
taus = np.round(np.linspace(0.05, 0.95, 19), 2)
miss = np.array([rates((p_blocked > t).astype(int))[1] for t in taus])
stop = np.array([rates((p_blocked > t).astype(int))[2] for t in taus])
cost = C_MISS * miss + C_STOP * stop
TAU = float(taus[np.argmin(cost)])
P_MISS, P_FALSE = rates((p_blocked > TAU).astype(int))[1:]
print(f"cheapest threshold {TAU:.2f}: missed {100 * P_MISS:.1f} %, false stops {100 * P_FALSE:.1f} %")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(taus, 100 * miss, "o-", ms=3, label="missed obstacles")
ax.plot(taus, 100 * stop, "s-", ms=3, label="false stops")
ax.axvline(TAU, color="k", ls="--", lw=1)
ax.set_xlabel(r"threshold $\tau$ on $p_\mathrm{b}$"); ax.set_ylabel("% of frames"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** Lowering $\tau$ trades missed obstacles for false
stops. On these frames the network misses none up to a threshold of about
0.8, so the cheapest threshold is the one with the fewest false stops among
those: a few per cent of free frames. That answers *is the path blocked*.

---

## 6 · The avoidance strategies

**What do I do about it?** Three strategies, each needing more from the
perception than the one before (L11.2, *Deciding the Action*):

| strategy | when blocked | what it must be told |
|---|---|---|
| **stop** | stand still until the path is free | free or blocked |
| **steer around** | aim at a lane 0.30 m to the side away from the obstacle, slowed to 60 %, until 0.5 m past it | free or blocked, and which side |
| **potential field** | aim, one aim distance ahead, at the lowest point of $U$ | where each obstacle stands |

**The potential field** adds a push away from every obstacle within range to a
pull back to the line (L11.2, *Steering Around*):

$$U(y) = y^2 + \sum_i \frac{K}{d_i(y)}, \qquad y^{*} = \arg\min_y U(y)$$

$y$ is the sideways position one aim distance $\ell$ ahead, $d_i(y)$ the gap
there between the car's edge and obstacle $i$ (floored at 1 cm), and $K$ =
0.03 m the strength of the push. The car aims at $y^{*}$.

`pb.avoid(strategy, f)` drives the course with a decision `f` times a second;
`strategy` is `"stop"`, `"steer"`, or a function `aim(x, near)` that returns
$y^{*}$ for a car at `x` and the obstacles `near` it (rows of $(x, y, r)$).

**TODO 3** is that function.

In [ ]:
def field_aim(x, near, K=pb.K_FIELD):
    # the sideways position to aim at, one aim distance ahead of a car at x
    ahead = x + pb.AIM                                        # where the aim point lies along the course
    ys = np.linspace(-0.6, 0.6, 241)                          # candidate sideways positions, 5 mm apart
    # TODO 3: U(y) = y**2 + sum K / d_i(y) on the candidates ys, and the y where it is lowest.
    #
    #   U = ys ** 2                                               # the pull back to the line
    #   for (ox, oy, r) in near:                                  # one push for every obstacle within range
    #       d = np.maximum(np.hypot(ahead - ox, ys - oy) - r - pb.CAR_HALF, 0.01)   # the gap from the car's edge
    #       U = U + K / d
    #   return float(ys[np.argmin(U)])                            # where U is lowest
    raise NotImplementedError("Write the potential field")

In [ ]:
STRATEGIES = {"stop": "stop", "steer around": "steer", "potential field": field_aim}
runs = {}
print(f"{'strategy':18s}{'decisions':>11s}{'time to B':>12s}{'smallest gap':>15s}")
for name, s in STRATEGIES.items():
    for f in (5, 10, 30):
        r = pb.avoid(s, f)                                    # perfect perception: the true free or blocked
        runs[name, f] = r
        when = f"{r['time']:.2f} s" if r["reached"] else "never"
        print(f"  {name:16s}{f:8d} Hz{when:>12s}{100 * r['clearance']:12.1f} cm" + ("   <- collision" if r["clearance"] < 0 else ""))

# the same at 10 Hz, perceived by section 5's network: its own missed and false rates, 20 runs each
print(f"with the network's errors at threshold {TAU:.2f} (missed {100 * P_MISS:.1f} %, false {100 * P_FALSE:.1f} %), 20 runs at 10 Hz:")
cnn_runs = {}
for name, s in list(STRATEGIES.items())[1:]:
    rr = [pb.avoid(s, 10, p_miss=P_MISS, p_false=P_FALSE, seed=k) for k in range(20)]
    cnn_runs[name] = rr
    t = [r["time"] for r in rr if r["reached"]]
    print(f"  {name:16s} collisions {sum(r['clearance'] < 0 for r in rr):2d} of 20, mean time to B {np.mean(t):.2f} s")

fig, ax = plt.subplots(figsize=(11, 3))
pb.draw_course(ax, paths={name: runs[name, 10]["path"] for name in STRATEGIES})
ax.set_title("the three strategies at 10 decisions a second", fontsize=9)
plt.tight_layout(); plt.savefig(cc.output_path("Ex11_report_2.png"), dpi=150); plt.show()

**What you should see.**

* **Stop never reaches B**: the first obstacle stands on the line and does not
  move, so the car stands in front of it until the 30 s run ends.
* **Steer around reaches B in about 8.7 s** with about 2 cm to spare at 10 Hz,
  and touches the second obstacle at 5 Hz: it holds a fixed side lane and
  turns only when the obstacle is already within 0.7 m.
* **The potential field reaches B in about 6.3 s** with about 10 cm to spare:
  it knows where each obstacle stands, never slows, and bends the path only
  as much as the gap needs.

With the network's errors in place of perfect perception, nothing changes
much: it misses no obstacle at this threshold, and a few false stops cost a
few tenths of a second. That answers *what do I do about it*: the strategy
decides what the perception must deliver, and the richer answer - where, not
only whether - is what bought the 2.4 s.

---

## 7 · The loop rate, the GPU and fewer bits

**The blind distance on a lap.** `pb.drive(f, v, error)` drives the car round
a stadium track of 7.20 m (straights 1.4 m, turns of 0.7 m radius, both
assumed) following the line, with a steering decision $f$ times a second from
a target seen with section 4's error. Computing a decision takes one period, so
the wheels follow a command that is one frame old. The cell reports the
largest distance from the centreline; half the course's narrowest gap is
17.5 cm (assumed).

**The stopping distance.** From the frame that shows an obstacle to standing
still, the car drives one blind distance, then brakes as hard as the tyres
allow (L11.2, *Stopping Distance*). Both must fit inside the 0.7 m the camera
judges:

$$d_{\mathrm{stop}} = \frac{v}{f} + \frac{v^2}{2\mu g} \leq R$$

with a friction coefficient $\mu$ = 0.6, **assumed**. The cell finds the
largest safe speed at three loop rates.

**The GPU** (L11.1, *Why a GPU*). Every output of a convolution is its own sum
of products over a small window, independent of the others, so a GPU computes
them at the same time. The cell counts the multiply-adds of one frame and times
one frame and a batch of 256 on the CPU and, if this runtime has one, on the
GPU.

**Fewer bits** (L11.2, *Deployment on the Car*). On a car the network's
weights are stored in fewer bits so that a decision takes less time and
memory. `pb.round_weights(net, b)` rounds every weight onto $2^b$ levels; the
cell measures what that does to both networks.

In [ ]:
print(f"{'speed':>8s}{'decisions':>12s}{'blind':>9s}{'furthest from the centreline':>32s}")
noise = err_net[1]                                            # the deliberate network's error, off the line
loop = []
for v, f in ((1.0, 30), (2.0, 30), (2.0, 20), (1.0, 10), (2.0, 15), (2.0, 12), (2.0, 10), (1.0, 5)):
    dev = pb.drive(f, v, noise)                               # two laps of the bicycle model
    loop.append((v / f, dev))
    print(f"{v:6.1f} m/s{f:9d} Hz{100 * v / f:7.1f} cm{100 * dev:27.2f} cm" + ("   <- off the course" if dev > pb.HALF_CLEARANCE else ""))

MU, G = 0.6, 9.81                                             # friction coefficient ASSUMED; gravity
speeds = np.linspace(0, 5, 5001)
safe = {f: float(speeds[speeds / f + speeds ** 2 / (2 * MU * G) <= pb.RANGE].max()) for f in (5, 10, 30)}
print("largest safe speed within the camera's 0.7 m: " + ", ".join(f"{s:.2f} m/s at {f} Hz" for f, s in safe.items()))

In [ ]:
def multiply_adds(model):
    # count the multiply-adds of one frame, layer by layer
    count = []
    def hook(layer, inp, out):
        if isinstance(layer, torch.nn.Conv2d):
            count.append(out[0].numel() * layer.in_channels * layer.kernel_size[0] * layer.kernel_size[1])   # every output: in-channels x kernel area
        elif isinstance(layer, torch.nn.Linear):             # every output: one product per input
            count.append(layer.in_features * layer.out_features)
    hooks = [m.register_forward_hook(hook) for m in model.modules()]   # listen to every layer
    with torch.no_grad():  # no gradient tracking: evaluation only
        model(torch.zeros(1, 1, pb.H, pb.W, device=next(model.parameters()).device))
    for h in hooks:
        h.remove()
    return sum(count)

print(f"one frame through the steering network: {multiply_adds(net):,} multiply-adds")
frames_cpu = to_tensor(TEST['off-line'][0][:256])             # 256 frames on the CPU
devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
gpu_time = {}
for dev in devices:
    m = copy.deepcopy(net).to(dev).eval()                     # the same network on this device
    one, batch = frames_cpu[:1].to(dev), frames_cpu.to(dev)
    sync = torch.cuda.synchronize if dev == "cuda" else (lambda: None)   # wait for the GPU before reading the clock
    with torch.no_grad():  # no gradient tracking: evaluation only
        t1 = middle(lambda: (m(one), sync()))
        t256 = middle(lambda: (m(batch), sync()))
    gpu_time[dev] = (t1, t256)
    print(f"  {dev:4s}: one frame {1000 * t1:.2f} ms, 256 frames {1000 * t256:.2f} ms, {1000 * t256 / 256:.3f} ms a frame")
if len(devices) == 1:
    print("  no GPU on this runtime: Runtime -> Change runtime type -> T4 GPU, and run the notebook again")

bits = (8, 6, 5, 4, 3, 2)
err_bits = [mean_error(run(pb.round_weights(net, b)), "off-line") for b in bits]   # the steering network, its weights rounded
acc_bits = []
for b in bits:
    with torch.no_grad():  # no gradient tracking: evaluation only
        p = to_numpy(torch.softmax(pb.round_weights(obs_net, b)(to_tensor(xt).to(DEVICE)), 1))[:, 1]   # p_b, weights in b bits
    acc_bits.append(rates((p > TAU).astype(int))[0])
print("weights in " + ", ".join(str(b) for b in bits) + " bits:")
print("  steering, mean error off the line: " + ", ".join(f"{e:.3f}" for e in err_bits) + f"  (as trained: {err_net[1]:.3f})")
print("  obstacles, right: " + ", ".join(f"{100 * a:.1f} %" for a in acc_bits) + f"  (as trained: {100 * rates((p_blocked > TAU).astype(int))[0]:.1f} %)")

**What you should see.**

* **The blind distance decides, not the speed or the loop rate alone.** The car
  stays within about 2 cm of the centreline up to 10 cm blind, is 3 cm off at
  13 cm, and leaves the course at 17 cm and beyond; 1 m/s at 10 Hz and 2 m/s
  at 20 Hz are both 10 cm blind and both hold the line.
* **The safe speed rises with the loop rate**: about 1.9 m/s at 5 Hz, 2.3 at
  10 Hz and 2.7 at 30 Hz within the 0.7 m the camera judges. The braking term
  does not depend on the network at all; the blind term is all the loop rate
  can buy.
* **One frame is about 0.65 million multiply-adds.** On a CPU one frame takes a
  fraction of a millisecond; on a GPU one frame alone is no faster, because
  starting the work costs more than the work, but a batch of 256 costs about
  as little a frame as the GPU's many cores allow. On a car the frames arrive
  one at a time, which is why the network is compiled for the device and its
  rate measured there, not on a laptop.
* **Fewer bits.** The steering network keeps its accuracy down to about 5
  bits and loses it at 3, because every rounding error lands in the answer.
  The obstacle network keeps its accuracy longer: a class only changes when
  the rounding moves $p_{\mathrm{b}}$ across the threshold.

---

## 8 · What the notebook says

* **The data decided the result, not the network.** Two identical steering
  networks, the same number of frames and the same training time: one fails
  beside the line and in glare, the other does not. Frames from where the car
  should not be are the ones that teach it to come back.
* **A rule fails on what its designer did not think of.** The line detector
  took glare for the line; the dark-pixel rule took shadows for obstacles and
  missed boxes with a light face. Both need no data, are the fastest, and are
  exact on the frames they were designed for.
* **The threshold is a decision about costs, not a property of the network.**
  With a collision costing a hundred false stops, it went well below one half.
* **The strategy decides what the perception must deliver.** Stopping needs
  only free or blocked and never reached B; the potential field needs where
  each obstacle stands and was fastest, with the widest gap.
* **The loop rate is a speed limit.** The blind distance limits how well the
  car follows the line and how fast it may drive towards an obstacle.
* **None of this is the real car.** The frames are drawn; the course, the
  car's size, its friction and its loop rate are assumed; the timings are this
  runtime's. The JetRacer in the lab, and the mini projects, are where they
  are measured.

---

## 9 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex11_report.md` with the results of sections 3 to 7 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L11.1 and L11.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The classical line detector and the network were scored on the same held-out frames. Where did each meet the agreed error, where did each fail, and why does the network not take the glare for the line? (-> L11.1 Q1)": """
""",
    "The two steering networks differ only in the frames they were trained on. What did the first one never see, why did its own held-out error not show the problem, and what would you collect on a real car because of it? (-> L11.1 Q2)": """
""",
    "On the lap, 1 m/s at 10 Hz and 2 m/s at 20 Hz gave the same result. What is the quantity that decides, at what value did the car leave the course, and what are the two ways to reduce it? (-> L11.1 Q3)": """
""",
    "How many multiply-adds does one frame take, how did one frame and a batch of 256 compare on the CPU and on the GPU, and what would you measure on the car before trusting a loop rate? (-> L11.1 Q4)": """
""",
    "What did the dark-pixel rule call blocked that was not, and what did it miss? What does the network take and return, and what does the cross-entropy reward? (-> L11.2 Q1)": """
""",
    "Which threshold did the expected cost choose, and how would it move if a collision cost ten false stops instead of a hundred? (-> L11.2 Q2)": """
""",
    "Why did stopping never reach B, and what does the potential field need to be told that steering around does not? (-> L11.2 Q3)": """
""",
    "What is the largest safe speed at 5, 10 and 30 decisions a second, and which of the two terms of the stopping distance does a faster network shorten? (-> L11.2 Q4)": """
""",
    "To conclude: which avoidance strategy reaches B fastest without a collision, and what limits its speed: the errors of the networks, the loop rate, or the distance needed to react? (-> L11.1 and L11.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| finding the line: mean error on 1000 held-out frames | line | off-line | glare | one frame |", "|---|---|---|---|---|",
            f"| classical detector, 12 x 16 | {classic[K][0]:.3f} | {classic[K][1]:.3f} | {classic[K][2]:.3f} | {1000 * t_classic:.2f} ms |",
            f"| network, trained along the line | {err_line[0]:.3f} | {err_line[1]:.3f} | {err_line[2]:.3f} | {1000 * t_net:.2f} ms |",
            f"| network, trained deliberately | {err_net[0]:.3f} | {err_net[1]:.3f} | {err_net[2]:.3f} | {1000 * t_net:.2f} ms |",
            "", "| free or blocked: 2000 held-out frames | right | missed | false stops |", "|---|---|---|---|",
            *[f"| {name} | {100 * r[0]:.1f} % | {100 * r[1]:.1f} % | {100 * r[2]:.1f} % |" for name, r in
              (("dark-pixel rule", rates(rule)), ("network, threshold 0.5", rates(cnn)), (f"network, threshold {TAU:.2f}", rates((p_blocked > TAU).astype(int))))],
            "", "| strategy, 10 Hz | time to B | smallest gap |", "|---|---|---|",
            *[f"| {name} | {runs[name, 10]['time']:.2f} s | {100 * runs[name, 10]['clearance']:.1f} cm |" if runs[name, 10]["reached"]
              else f"| {name} | never | {100 * runs[name, 10]['clearance']:.1f} cm |" for name in STRATEGIES],
            "", "Blind distance against the furthest the car got from the centreline: "
            + ", ".join(f"{100 * d:.1f} cm -> {100 * e:.2f} cm" for d, e in sorted(loop)),
            "Largest safe speed: " + ", ".join(f"{s:.2f} m/s at {f} Hz" for f, s in safe.items()) + " (friction coefficient 0.6 assumed)",
            f"One frame: {multiply_adds(net):,} multiply-adds; " + "; ".join(f"{d}: one frame {1000 * a:.2f} ms, 256 frames {1000 * b:.2f} ms" for d, (a, b) in gpu_time.items()),
            "Steering network, weights in " + ", ".join(str(b) for b in bits) + " bits: mean error " + ", ".join(f"{e:.3f}" for e in err_bits),
            "", "The frames are drawn; the course, the car and its friction are assumed."]
    out = ["# Ex_11 — Vision-Based Navigation, from A to B", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex11_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures of sections 4 and 6, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex11_report.md into Ex11_report.pdf, with any figure
# saved as Ex11_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex11_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex11_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex11_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex11_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 10 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP11A · The camera with a LiDAR

**The problem.** The notebook's car decides from one camera. A camera does not see an obstacle as
light as the floor, and calls a shadow an obstacle. A LiDAR measures the range
at every bearing round the car, but only in its scan plane, so an obstacle
lower than that plane is invisible to it, and it scans five times a second
where the camera gives thirty frames. Here one network reads both and drives
from A to B.

| | |
|---|---|
| courses | 6 m from A to B, four obstacles at least 0.9 m apart, two shadows; random, seeded |
| obstacles | radius 6 to 10 cm, of three kinds: a dark box 20 cm tall (every sensor sees it), a grey box as light as the floor (the camera barely sees it), a dark block 5 cm low (under the LiDAR's plane) |
| camera | the notebook's 48 × 64 grey frame of the floor 0 to 1.56 m ahead, 10 decisions a second |
| LiDAR | RPLIDAR A1: 360 ranges, one a degree, 0.15 to 12 m, 5.5 scans a second (vendor); scan plane 12 cm high, noise 1 % of the range (assumed); no return reads 0 |
| car | the notebook's: 1 m/s, the steer-around rule, half-width 10 cm, camera range 0.7 m (assumed) |

The vendor's figures are checked against the data sheet before the data is
released. You have the car and the sensor: the last requirement is a run on
the floor.

**Why a plain PINN is not enough.** Neither sensor is enough, and their errors are of different kinds: the camera
misses what the LiDAR sees and sees what the LiDAR misses, at different rates.
A rule that ORs the two inherits both sets of false stops, and still does not
say where an obstacle the camera alone sees stands.

**What you build.** One network with two branches - convolutions over the frame, a 1-D
convolution over the scan, which wraps round at 360° - joined into two heads:
blocked or free (cross-entropy), and where the obstacle stands, forward and
sideways (squared error). You put it in the loop of `problem.run_course` and
drive the held-out courses.

**The ground truth you get.** `MP11A_truth.npz`, from `tools/miniprojects/ex11_truth.py`, computed by
Ex11's `problem.py`: 6000 poses on 200 courses for training and 1500 on 50
held-out courses, each with its frame, scan, the true label, where the
obstacle stands and its kind; 20 more held-out courses to drive, with the time
and gap of a perfect perception. The labels and ranges are exact geometry; the
drive is converged in its time step (4, 2 and 1 ms agree to 2 ms in time).

**Worked example.** 

| classical rule, 1500 held-out poses | missed: dark box | grey box | low block | stops for a shadow | right |
|---|---|---|---|---|---|
| camera, dark pixels in the lane | 14.2 % | 77.5 % | 6.7 % | 100 % | 73.1 % |
| LiDAR, a return in the lane | 12.3 % | 16.2 % | 100 % | 23.3 % | 74.6 % |
| camera OR LiDAR | 0.0 % | 11.8 % | 6.7 % | 100 % | 76.8 % |

Driven on the 20 held-out courses, perfect perception reaches B in 8.90 s on
average with at least 3.3 cm to spare; the LiDAR rule collides on 17 courses,
camera OR LiDAR on 11. A first network with both branches, trained for a
minute on the training poses, missed 5, 11 and 5 % of the three kinds, placed
the obstacle within 8 cm forward and 4 cm sideways, and collided on 6 of the 20
courses: the month goes into closing that gap.

**Requirements.**

* On the held-out poses: at most **5 %** of each kind of obstacle missed, and at most **5 %** of the shadows called blocked.
* Where the obstacle stands within **4 cm**, forward and sideways (root mean square).
* On the 20 held-out courses: **no collision**, and a mean time to B within **10 %** of perfect perception's 8.90 s.
* On the car: the RPLIDAR on the JetRacer, one run from A to B on a taped course with one obstacle of each kind; the time and the smallest gap.

**What you hand in.** The missed obstacles per kind and the false stops against the classical rules,
the position error, the closed-loop times and gaps, what each sensor
contributed (the network with each branch removed), and the run on the car
with what differed from the simulation.

### MP11B · The camera with an ultrasonic sensor

**The problem.** An ultrasonic sensor answers with one number: the distance to the nearest
surface anywhere in a wide cone. It hears the grey box and the low block that
fool the camera and the LiDAR, and it is cheap, but it cannot say whether the
obstacle stands to the left or to the right - which is what the car needs to
pass it. The camera can. Here one network reads both.

| | |
|---|---|
| courses, obstacles, camera, car | as MP11A |
| ultrasonic | DFRobot URM09 on I2C: 2 to 500 cm, 1 cm steps, accuracy 1 % (vendor); cone ±30°, 20 readings a second, mounted 4 cm high (assumed); 5.00 m when nothing answers |

The vendor's figures are checked against the data sheet before the data is
released. You have the car and the sensor: the last requirement is a run on
the floor.

**Why a plain PINN is not enough.** The two sensors answer different questions - how far, and where - and neither
answers both. The sensor's range comes from anything in its cone, inside the
lane or not, so the network has to learn when to believe it.

**What you build.** One network: convolutions over the frame, with the ultrasonic range joined in
before the dense layers, and two heads, blocked or free and where the obstacle
stands. You put it in the loop of `problem.run_course` and drive the held-out
courses.

**The ground truth you get.** `MP11B_truth.npz`, from the same script: the same poses and courses as
MP11A, with the ultrasonic range in place of the scan.

**Worked example.** 

| classical rule, 1500 held-out poses | missed: dark box | grey box | low block | stops for a shadow | right |
|---|---|---|---|---|---|
| camera, dark pixels in the lane | 14.2 % | 77.5 % | 6.7 % | 100 % | 73.1 % |
| ultrasonic, a range under 0.7 m | 19.6 % | 27.9 % | 15.2 % | 10.1 % | 87.1 % |
| camera OR ultrasonic | 2.7 % | 22.5 % | 3.0 % | 100 % | 79.1 % |

The ultrasonic rule is right most often, and still collides on all 20
held-out courses: it says blocked but not on which side, and the car, told
nothing, passes on the wrong one. A first network, trained for a minute on the
training poses, missed 7, 22 and 4 % of the three kinds and collided on 1 of
the 20 courses.

**Requirements.**

* On the held-out poses: at most **5 %** of each kind of obstacle missed, and at most **5 %** of the shadows called blocked.
* Where the obstacle stands within **5 cm**, forward and sideways (root mean square).
* On the 20 held-out courses: **no collision**, and a mean time to B within **10 %** of perfect perception's 8.90 s.
* On the car: the URM09 on the JetRacer's I2C bus, one run from A to B on a taped course with one obstacle of each kind; the time and the smallest gap.

**What you hand in.** The same as MP11A, and one paragraph comparing with it if anyone in the
class took MP11A: what a 360° scan buys over one range, on these obstacles.